In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [4]:
from typing import Optional
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg
from scipy.spatial.distance import cdist
from scipy.sparse.csgraph import dijkstra
from scipy.spatial.transform import Rotation

#from isomap_master.isomap import make_adjacency, isomap, plot_graph
from sklearn.decomposition import PCA

#from SampleMaker import make_sample_vectorized
from DownSampler import load_and_downsample
from SampleCombiner import combine_to_7d_tensor, combine_to_7d
from WeatheringScore import compute_weather_score
import time

#from IPython.display import display
#from IPython.display import clear_output


### Test Sample ###

In [ ]:
#X = make_sample_vectorized(50)

#X.shape

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def visualize_sample(data):
    """
    data: np.ndarray (H, W, 7)
    """

    assert data.ndim == 3 and data.shape[2] == 7

    diffuse = data[..., 0:3]
    specular = data[..., 3:6]
    roughness = data[..., 6]

    # -----------------------------
    # Clamp (안전)
    # -----------------------------
    diffuse = np.clip(diffuse, 0.0, 1.0)
    specular = np.clip(specular, 0.0, 1.0)
    roughness = np.clip(roughness, 0.0, 1.0)

    # -----------------------------
    # Plot
    # -----------------------------
    fig, axes = plt.subplots(1, 3, figsize=(9, 3))

    # Diffuse (RGB)
    axes[0].imshow(diffuse)
    axes[0].set_title("Diffuse (RGB)")
    axes[0].axis("off")

    # Specular (RGB)
    axes[1].imshow(specular)
    axes[1].set_title("Specular (RGB)")
    axes[1].axis("off")

    # Roughness (Heatmap)
    im = axes[2].imshow(roughness, cmap='jet')
    axes[2].set_title("Roughness")
    axes[2].axis("off")

    fig.colorbar(im, ax=axes[2])

    plt.tight_layout()
    plt.show()

In [ ]:
#visualize_sample(X)

### Test Sample Texture ###

In [6]:
DPath = "./Military_Trenches_Wall_Metal_Corrugated_03_yd0jdits_Mid_2K_BaseColor.png"
SPath = "./Military_Trenches_Wall_Metal_Corrugated_03_yd0jdits_Mid_2K_Specular.png"
RPath = "./Military_Trenches_Wall_Metal_Corrugated_03_yd0jdits_Mid_2K_Roughness.png"
D,S,R = load_and_downsample(DPath, SPath, RPath, 128)

X = combine_to_7d_tensor(D,S,R) # 2차원 이미지 형태
FX = combine_to_7d(D,S,R)       # 1차원 배열

visualize_sample(X)

FileNotFoundError: [Errno 2] No such file or directory: './Military_Trenches_Wall_Metal_Corrugated_03_yd0jdits_Mid_2K_BaseColor.png'

### Basic Manifold Generator ###

Build Isomap with K-nn rule, epsilon threshold rule.
Points that are far away are excluded from connection based on a value that is 1.5 times the average distance to k points.  

In [ ]:
import heapq
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import lil_matrix
from scipy.sparse.csgraph import dijkstra
from scipy.sparse.linalg import eigsh
import cupy as cp

from scipy.sparse import coo_matrix
import numpy as np


def normalize_features(X):

    X = X.astype(np.float32)

    mean = np.mean(X, axis=0, keepdims=True)
    std = np.std(X, axis=0, keepdims=True)

    return (X - mean) / (std + 1e-8)

def compute_distance_chunk(
    X_chunk,
    X_all
):

    dD = (
        X_chunk[:, None, 0:3] -
        X_all[None, :, 0:3]
    )

    dS = (
        X_chunk[:, None, 3:6] -
        X_all[None, :, 3:6]
    )

    rA = cp.sqrt(
        cp.clip(X_chunk[:, None, 6], 0, 1)
    )

    rB = cp.sqrt(
        cp.clip(X_all[None, :, 6], 0, 1)
    )

    dR = rA - rB

    avgR = 0.5 * (
        X_chunk[:, None, 6] +
        X_all[None, :, 6]
    )

    wD = (
        (1.0 - avgR) * 0.8 +
        avgR * 1.2
    )

    wS = (
        (1.0 - avgR) * 0.5 +
        avgR * 0.2
    )

    dist2 = (
        wD * cp.sum(dD * dD, axis=2) +
        wS * cp.sum(dS * dS, axis=2) +
        dR * dR
    )

    return cp.sqrt(
        cp.maximum(dist2, 1e-8)
    )


def build_sparse_knn_graph(
    X,
    K=8,
    epsilon_scale=1.5,
    chunk_size=1024
):

    X_gpu = cp.asarray(
        X,
        dtype=cp.float32
    )

    N = X.shape[0]

    edge_src = []
    edge_dst = []
    edge_weight = []

    for start in range(0, N, chunk_size):

        end = min(start + chunk_size, N)

        X_chunk = X_gpu[start:end]

        dist = compute_distance_chunk(
            X_chunk,
            X_gpu
        )
        rows = cp.arange(end - start)

        dist[rows, start:end] += 1e9

        knn_idx = cp.argpartition(
            dist,
            K,
            axis=1
        )[:, :K]

        knn_dist = cp.take_along_axis(
            dist,
            knn_idx,
            axis=1
        )
        dp = cp.mean(knn_dist, axis=1)

        epsilon = (
            epsilon_scale * dp
        )

        mask = (
            knn_dist <= epsilon[:, None]
        )

        src = cp.repeat(
            cp.arange(start, end),
            K
        )

        dst = knn_idx.reshape(-1)

        weight = knn_dist.reshape(-1)

        mask = mask.reshape(-1)

        src = src[mask]
        dst = dst[mask]
        weight = weight[mask]

        valid = src < dst

        src = src[valid]
        dst = dst[valid]
        weight = weight[valid]

        edge_src.append(cp.asnumpy(src))
        edge_dst.append(cp.asnumpy(dst))
        edge_weight.append(cp.asnumpy(weight))

    edge_src = np.concatenate(edge_src)
    edge_dst = np.concatenate(edge_dst)
    edge_weight = np.concatenate(edge_weight)

    return edge_src, edge_dst, edge_weight


def compute_geodesic_from_graph_cuda(
    edge_src,
    edge_dst,
    edge_weight,
    N
):

    """
    Parameters
    ----------
    edge_src : (E,)
    edge_dst : (E,)
    edge_weight : (E,)
    N : int

    Return
    ------
    TODO
    """

    rows = np.concatenate([edge_src, edge_dst])
    cols = np.concatenate([edge_dst, edge_src])
    data = np.concatenate([edge_weight, edge_weight])

    adj = coo_matrix(
        (data, (rows, cols)),
        shape=(N, N)
    )

    adj = adj.tocsr()

    geodesic = dijkstra(
        adj,
        directed=False
    )

    return geodesic

def mds_embedding_sparse(
    D,
    dim=3
):

    N = D.shape[0]
    print("D.shape[0] : " + str(N))

    D = D.copy()
    print("D: " + str(D))

    finite_mask = np.isfinite(D)

    D[~finite_mask] = np.max(
        D[finite_mask]
    )

    H = (
        np.eye(N) -
        np.ones((N, N)) / N
    )

    B = -0.5 * H @ (D**2) @ H

    B = np.nan_to_num(B)

    B = (B + B.T) * 0.5

    evals, evecs = eigsh(
        B,
        k=dim,
        which='LA'
    )

    idx = np.argsort(evals)[::-1]

    evals = evals[idx]
    evecs = evecs[:, idx]

    Z = (
        evecs *
        np.sqrt(np.maximum(evals, 0))
    )

    return Z



### Revised Graph Generator ###

The graph generation function actually in use is the function below.

In [ ]:
from sklearn.neighbors import NearestNeighbors


def build_weathering_graph(
    samples_7d,
    K=8,
    extra_ratio=4,
    epsilon_scale=1.5
):

    N = samples_7d.shape[0]

    weather_score = compute_weather_score(
        samples_7d
    )
    print("Weather Score : ")
    print(weather_score)

    extra_k = min(
        K * extra_ratio,
        N - 1
    )

    nn = NearestNeighbors(
        n_neighbors=extra_k + 1,
        algorithm='auto'
    )

    nn.fit(samples_7d)

    distances, indices = nn.kneighbors(
        samples_7d
    )

    # remove self
    distances = distances[:, 1:]
    indices = indices[:, 1:]
    
    print("Distances :")
    print(distances)
    print()
    print("Indices :")
    print(indices)


    edge_src = []
    edge_dst = []
    edge_weight = []

    for i in range(N):

        local_dists = distances[i]
        local_inds = indices[i]

        dp = np.mean(local_dists[:K])

        epsilon = (
            epsilon_scale * dp
        )

        score_i = weather_score[i]

        for dist, j in zip(local_dists, local_inds):

            if dist > epsilon:
                break

            score_j = weather_score[j]

            delta = score_j - score_i

            if delta >= 0:

                # forward progression
                progression_factor = (
                    1.0 - 0.35 * delta
                )

            else:

                # backward penalty
                progression_factor = (
                    1.0 + 1.5 * abs(delta)
                )

            weight = dist * progression_factor

            progression_bonus = (
                1.0 - delta
            )

            weight = (
                dist * progression_bonus
            )

            edge_src.append(i)
            edge_dst.append(j)
            edge_weight.append(weight)

    return (
        np.array(edge_src, dtype=np.int32),
        np.array(edge_dst, dtype=np.int32),
        np.array(edge_weight, dtype=np.float32),
        weather_score
    )

### Prototype Generator ### 

This is code for initial testing. It cannot be used due to significant performance flaws.

In [ ]:
def distance_7d(a, b):
    """
    a, b: (7,)
    [Dx,Dy,Dz, Sx,Sy,Sz, R]
    """

    avgR = 0.5 * (a[6] + b[6])

    w_diffuse = (1.0 - avgR) * 0.8 + avgR * 1.2
    w_specular = (1.0 - avgR) * 0.5 + avgR * 0.2
    w_roughness = 1.0

    dD = a[0:3] - b[0:3]
    dS = a[3:6] - b[3:6]

    rA = np.sqrt(a[6])
    rB = np.sqrt(b[6])
    dR = rA - rB

    sum_val = (
        w_diffuse * np.dot(dD, dD) +
        w_specular * np.dot(dS, dS) +
        w_roughness * (dR * dR)
    )

    return np.sqrt(sum_val)

def build_knn_graph(data_7d, K):
    """
    data_7d: (N, 7)
    return: adjacency list
            List[List[(neighbor_index, distance)]]
    """

    #nbrs = NearestNeighbors(n_neighbors=K+1, algorithm='auto').fit(data_7d)
    
    data_gpu = cp.asarray(data_7d)

    nbrs = NearestNeighbors(n_neighbors=K+1)
    nbrs.fit(data_gpu)
    
    distances, indices = nbrs.kneighbors(data_7d)

    N = data_7d.shape[0]
    graph = [[] for _ in range(N)]

    for i in range(N):

        # -----------------------------
        # 1. KNN 거리 (자기 자신 제외)
        # -----------------------------
        knn_dists = distances[i][1:K+1]
        knn_indices = indices[i][1:K+1]

        # -----------------------------
        # 2. dp 및 epsilon 계산
        # -----------------------------
        dp = np.mean(knn_dists)
        epsilon = 1.5 * dp

        # -----------------------------
        # 3. 🔥 KNN은 무조건 추가
        # -----------------------------
        for dist, j in zip(knn_dists, knn_indices):
            if i < j:
                graph[i].append((j, dist))
                graph[j].append((i, dist))

        # -----------------------------
        # 4. 🔥 epsilon 확장
        # (K 이후 이웃 탐색 필요)
        # -----------------------------
        # 추가 후보를 위해 더 많은 neighbor 조회
        extra_k = min(K * 4, N - 1)

        dists_ext, inds_ext = nbrs.kneighbors(data_7d[i].reshape(1, -1), n_neighbors=extra_k + 1)
        dists_ext = dists_ext[0][1:]
        inds_ext = inds_ext[0][1:]

        for dist, j in zip(dists_ext, inds_ext):

            # 이미 KNN에 포함된 경우 skip
            if j in knn_indices:
                continue

            if dist <= epsilon:
                if i < j:
                    graph[i].append((j, dist))
                    graph[j].append((i, dist))
            else:
                # 정렬되어 있으므로 이후는 다 탈락
                break

    return graph
    
def compute_geodesic_from_graph(graph, N):
    """
    graph: adjacency list [(j, dist), ...]
    """

    adj = lil_matrix((N, N))

    for i in range(N):
        for j, dist in graph[i]:
            adj[i, j] = dist
            adj[j, i] = dist

    return dijkstra(adj, directed=False)


def mds_embedding(D, dim=3):
    """
    d: (N,N) geodesic distance matrix
    """

    N = D.shape[0]

        # INF 처리
    D = D.copy()
    D[np.isinf(D)] = np.max(D[~np.isinf(D)])

        # Double centering
    H = np.eye(N) - np.ones((N, N)) / N
    B = -0.5 * H @ (D**2) @ H

        # 안정화
    B = np.nan_to_num(B)
    B = (B + B.T) * 0.5

        # Eigen decomposition (dense)
    evals, evecs = np.linalg.eigh(B)

    idx = np.argsort(evals)[::-1]
    evals = evals[idx][:dim]
    evecs = evecs[:, idx][:, :dim]

    Z = evecs * np.sqrt(np.maximum(evals, 0))

    return Z



# test sample shape :(64, 64, 7)
# total execution time : 120s

# test sample shape :(128, 128, 7)
# total execution time : inf



### GPU accelerated generation ###

This is a generator that introduces GPU acceleration using CuPy.
Since the introduction of CuPy, results are generated even with 128 * 128 data sampling.

Below is the result of performing MDS dimensionality reduction based on geodesic distances after generating a manifold graph.

Depending on the process state, it may be about 1.5 to 1.7 times slower than optimal performance.

In [ ]:
X = normalize_features(FX)
# test sample shape : (128, 128, 7)

### Generate Isomap Adjacent Graph with K-nn / adaptive epsilon rule ###

In [ ]:
start = time.perf_counter()
edge_src, edge_dst, edge_weight, scores = build_weathering_graph(X)
print("KNN Start")
end = time.perf_counter()
print("KNN Done")
print("Execution Time : " + str(end-start))
# execution time : 0.3~0.7s

### Export Geodesic Distances For all points with Dijkstra ###

In [ ]:
start = end


print("Geodesic Start")
D = compute_geodesic_from_graph_cuda(
    edge_src,
    edge_dst,
    edge_weight,
    X.shape[0]
)
end = time.perf_counter()
print("Execution Time : " + str(end-start))
print("Geodesic Done")
# execution time : 60~121s

### Generate MDS Dimentionality Reduction Graph For Directed Weathering ###

In [ ]:
start = end


print("MDS Start")
Z = mds_embedding_sparse(D)
end = time.perf_counter()
print("Execution Time : " + str(end-start))
print("MDS Done")
# execution time : 70~92s

### Visualization ###

This is the result of visualizing the generated MDS dimensionality reduction graph. The dimensionality of MDS is 3.

In [ ]:
def visualize_final(Z, data_7d):
    roughness = data_7d[:, 6]
    colors = data_7d[:, 0:3]

    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')

    ax.scatter(
        Z[:, 0],
        Z[:, 1],
        Z[:, 2],
        c=colors,
        s=1
    )

    plt.show()

In [ ]:
visualize_final(Z, FX)

### Analysis ###

Each point is displayed in the color corresponding to the Base Color at the location reconstructed through MDS dimensionality reduction.

Dots of similar colors form clusters and show a pattern of gradual change.

I believe that a single trajectory reflecting the time axis of weathering can be extracted from a graph of this quality.

### Texture Reconstruction ###



In [ ]:

# Function For Recontruction Projected Point
def reconstruct_from_mds(samples_7d, embedded, query_points, K=8):
    """
    samples_7d: (N,7)
    embedded: (N,3)
    query_points: (N,3)
    """

    N = samples_7d.shape[0]
    out = np.zeros_like(samples_7d)

    nbrs = NearestNeighbors(n_neighbors=K).fit(embedded)

    for qi in range(N):
        q = query_points[qi].reshape(1, -1)

        dists, indices = nbrs.kneighbors(q)
        indices = indices[0]

        Z = embedded[indices] - q  # (K,3)

        # Gram matrix
        C = Z @ Z.T

        # regularization
        trace = np.trace(C)
        C += np.eye(K) * trace * 1e-3

        # solve Cw = 1
        ones = np.ones(K)
        w = np.linalg.solve(C, ones)

        w /= (np.sum(w) + 1e-8)

        # reconstruction
        out[qi] = np.sum(samples_7d[indices] * w[:, None], axis=0)

    # clamp
    out[:, :6] = np.clip(out[:, :6], 0, 1)
    out[:, 6] = np.clip(out[:, 6], 0, 1)

    return out

def build_weathering_sequence(samples_7d, embedded, H, W, max_step=20):
    N = samples_7d.shape[0]

    # -----------------------------
    # 1. PCA (1D axis)
    # -----------------------------
    mean = embedded.mean(axis=0)
    X = embedded - mean

    # power iteration
    axis = np.array([1.0, 0.0, 0.0])
    for _ in range(20):
        axis = X.T @ (X @ axis)
        axis /= (np.linalg.norm(axis) + 1e-8)

    # -----------------------------
    # 2. t 계산
    # -----------------------------
    t_vals = X @ axis

    t_min, t_max = t_vals.min(), t_vals.max()
    t_vals = (t_vals - t_min) / (t_max - t_min + 1e-8)
    #t_vals = 1.0 - t_vals

    steps = []

    # -----------------------------
    # 3. step loop
    # -----------------------------
    for step in range(max_step):
        alpha = step / (max_step - 1)

        query_points = np.zeros_like(embedded)

        for i in range(N):
            target_t = t_vals[i] + alpha * (1.0 - t_vals[i])

            # closest t
            j = np.argmin(np.abs(t_vals - target_t))
            query_points[i] = embedded[j]

        reconstructed = reconstruct_from_mds(
            samples_7d,
            embedded,
            query_points,
            K=8
        )
        steps.append(reconstructed.reshape(H, W, 7))
       
    return steps

In [ ]:
import matplotlib.pyplot as plt


def visualize_step(step_data, step_idx):
    """
    step_data: (H, W, 7)
    """

    diffuse = step_data[:, :, 0:3]
    specular = step_data[:, :, 3:6]
    roughness = step_data[:, :, 6]

    fig, axs = plt.subplots(1, 3, figsize=(12, 4))

    axs[0].imshow(diffuse)
    axs[0].set_title(f"Diffuse (Step {step_idx})")

    axs[1].imshow(specular)
    axs[1].set_title("Specular")

    axs[2].imshow(roughness, cmap='gray')
    axs[2].set_title("Roughness")

    for ax in axs:
        ax.axis('off')

    plt.show()

    import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display


# Visulizer With Slider
def visualize_weathering_sequence(
    steps
):

    max_step = len(steps) - 1

    fig, axs = plt.subplots(
        1,
        3,
         figsize=(9, 3)
    )

    plt.tight_layout()


    diffuse = steps[0][:, :, 0:3]
    specular = steps[0][:, :, 3:6]
    roughness = steps[0][:, :, 6]

    im0 = axs[0].imshow(diffuse)
    axs[0].set_title("Diffuse")

    im1 = axs[1].imshow(specular)
    axs[1].set_title("Specular")

    im2 = axs[2].imshow(
        roughness,
        cmap='gray',
        vmin=0,
        vmax=1
    )
    axs[2].set_title("Roughness")

    for ax in axs:
        ax.axis("off")


    def update(step_idx):

        step_data = steps[step_idx]

        diffuse = step_data[:, :, 0:3]
        specular = step_data[:, :, 3:6]
        roughness = step_data[:, :, 6]

        im0.set_data(diffuse)
        im1.set_data(specular)
        im2.set_data(roughness)

        fig.suptitle(
            f"Weathering Step {step_idx}",
            fontsize=16
        )

        fig.canvas.draw_idle()


    slider = widgets.IntSlider(
        value=0,
        min=0,
        max=max_step,
        step=1,
        description='Step:',
        continuous_update=True
    )

    widgets.interact(
        update,
        step_idx=slider
    )

    plt.show()

### Weathering sequence only with MDS dimensionality reduction ###

In [ ]:
steps = build_weathering_sequence(FX, Z, H=128, W=128, max_step=20)
visualize_weathering_sequence(steps)

In [ ]:
from scipy.sparse.csgraph import shortest_path

def project_to_trajectory(
    embedded,
    trajectory
):

    """
    embedded:   (N,3)
    trajectory: (M,3)

    return:
        nearest_idx: (N,)
    """

    diff = (
        embedded[:, None, :] -
        trajectory[None, :, :]
    )

    dist2 = np.sum(diff * diff, axis=2)

    nearest_idx = np.argmin(dist2, axis=1)

    return nearest_idx



def sample_forward_trajectory(
    trajectory,
    nearest_idx,
    alpha
):

    """
    alpha: [0,1]
    """

    M = trajectory.shape[0]

    forward_idx = (
        nearest_idx +
        alpha * (M - 1 - nearest_idx)
    )

    forward_idx = np.clip(
        forward_idx.astype(np.int32),
        0,
        M - 1
    )

    return trajectory[forward_idx]

def reconstruct_from_embedding(
    samples_7d,
    embedded,
    query_points,
    K=8
):

    nbrs = NearestNeighbors(
        n_neighbors=K
    )

    nbrs.fit(embedded)

    dists, indices = nbrs.kneighbors(
        query_points
    )

    N = query_points.shape[0]

    out = np.zeros(
        (N,7),
        dtype=np.float32
    )

    for i in range(N):

        idx = indices[i]

        Z = (
            embedded[idx] -
            query_points[i]
        )

        C = Z @ Z.T

        trace = np.trace(C)

        C += (
            np.eye(K) *
            trace *
            1e-3
        )

        w = np.linalg.solve(
            C,
            np.ones(K)
        )

        w /= (
            np.sum(w) + 1e-8
        )

        out[i] = np.sum(
            samples_7d[idx] *
            w[:, None],
            axis=0
        )

    if False:
        out[:, :6] = np.clip(out[:, :6], 0, 1)

        out[:, 6] = np.clip(
            out[:, 6],
            0,
            1
        )
    out[:, :7] = np.clip(out[:, :7], 0, 1)

    return out

def build_weathering_sequence_trajectory(
    samples_7d: np.ndarray,
    embedded: np.ndarray,
    trajectory: np.ndarray, # TODO: Check type annoatiaon
    H: int,
    W: int,
    max_step: Optional[int]=20
) -> np.ndarray: # Search keywork: python type annotation

    N = embedded.shape[0]


    nearest_idx = project_to_trajectory(
        embedded,
        trajectory
    )

    steps = []

    for step in range(max_step):

        alpha = (
            step /
            (max_step - 1)
        )

        query_points = sample_forward_trajectory(
            trajectory,
            nearest_idx,
            alpha
        )

        reconstructed = reconstruct_from_embedding(
            samples_7d,
            embedded,
            query_points,
            K=8
        )

        steps.append(
            reconstructed.reshape(
                H,
                W,
                7
            )
        )

    return steps

In [ ]:
def visualize_trajectory(
    embedded,
    trajectory,
    elev=30,
    azim=120
):

    fig = plt.figure(figsize=(8,8))

    ax = fig.add_subplot(
        111,
        projection='3d'
    )

    # =====================================================
    # 1. manifold scatter
    # =====================================================

    ax.scatter(
        embedded[:,0],
        embedded[:,1],
        embedded[:,2],
        s=2,
        alpha=0.3
    )

    # =====================================================
    # 2. trajectory line
    # =====================================================

    ax.plot(
        trajectory[:,0],
        trajectory[:,1],
        trajectory[:,2],
        linewidth=4
    )

    # =====================================================
    # 3. start / end point
    # =====================================================

    ax.scatter(
        trajectory[0,0],
        trajectory[0,1],
        trajectory[0,2],
        s=100,
        marker='o'
    )

    ax.scatter(
        trajectory[-1,0],
        trajectory[-1,1],
        trajectory[-1,2],
        s=100,
        marker='^'
    )

    ax.view_init(
        elev=elev,
        azim=azim
    )

    plt.show()



In [ ]:
def resample_trajectory(
    trajectory_points,
    num_points=256
):

    seg = (
        trajectory_points[1:]
        - trajectory_points[:-1]
    )

    seg_len = np.linalg.norm(
        seg,
        axis=1
    )

    cumulative = np.concatenate([
        [0],
        np.cumsum(seg_len)
    ])

    total_len = cumulative[-1]

    sample_t = np.linspace(
        0,
        total_len,
        num_points
    )

    out = np.zeros(
        (num_points, 3),
        dtype=np.float32
    )

    for i, t in enumerate(sample_t):

        idx = np.searchsorted(
            cumulative,
            t
        ) - 1

        idx = np.clip(
            idx,
            0,
            len(seg_len)-1
        )

        t0 = cumulative[idx]
        t1 = cumulative[idx + 1]

        alpha = (
            (t - t0)
            / (t1 - t0 + 1e-8)
        )

        p0 = trajectory_points[idx]
        p1 = trajectory_points[idx + 1]

        out[i] = (
            p0 * (1 - alpha)
            + p1 * alpha
        )

    return out

def build_weathering_trajectory(
    embedded,
    edge_src,
    edge_dst,
    edge_weight,
    start_idx,
    end_idx
):

    """
    Parameters
    ----------
    embedded : (N,3)

    edge_src : (E,)
    edge_dst : (E,)
    edge_weight : (E,)

    start_idx : int
    end_idx : int

    Returns
    -------
    trajectory_points : (T,3)
    trajectory_indices : (T,)
    """

    N = embedded.shape[0]


    graph = coo_matrix(
        (
            edge_weight,
            (edge_src, edge_dst)
        ),
        shape=(N, N)
    ).tocsr()

    dist, predecessors = dijkstra(
        graph,
        directed=True,
        indices=start_idx,
        return_predecessors=True
    )

    path = []

    current = end_idx

    while current != start_idx:

        if current == -9999:
            raise ValueError(
                "Trajectory path disconnected"
            )

        path.append(current)

        current = predecessors[current]

    path.append(start_idx)

    path.reverse()

    path = np.array(path)

    trajectory_points = embedded[path]

    return trajectory_points, path

### Visualize Trajectory on MDS Graph ###

In [ ]:
start_idx = np.argmin(scores)
end_idx = np.argmax(scores)

trajectory_points, trajectory_nodes = (
    build_weathering_trajectory(
        embedded=Z,
        edge_src=edge_src,
        edge_dst=edge_dst,
        edge_weight=edge_weight,
        start_idx=start_idx,
        end_idx=end_idx
    )
)

trajectory = resample_trajectory(
    trajectory_points,
    num_points=256
)

visualize_trajectory(Z, trajectory)

### Visualize Weathering sequence w/ trajectory projection and reconstruction ### 

In [ ]:

steps = build_weathering_sequence_trajectory(FX, Z, trajectory, H=128, W=128, max_step=20)
visualize_weathering_sequence(steps)

In [ ]:
def compute_weather_score_new(
    diffuse,
    specular,
    roughness
):
    """
    diffuse  : (N,3)
    specular : (N,3)
    roughness: (N,)
    """

    # -----------------------------------------
    # 1. Rust-like darkness
    # -----------------------------------------

    # WARN: Use consistent convention throught the whole project
    # TODO: reference: choose one from followings:
    # * https://scikit-image.org/docs/stable/auto_examples/color_exposure/plot_rgb_to_gray.html
    # * https://kr.mathworks.com/help/matlab/ref/rgb2gray.html
    # * If you use wikipedia page, please clarify which section your are referring 
    #   (e.g. https:/wikipedia.org/{document_name}#SectionName)
    luminance = (
        diffuse[:, 0] * 0.299 +
        diffuse[:, 1] * 0.587 +
        diffuse[:, 2] * 0.114
    )

    darkness = 1.0 - luminance

    # -----------------------------------------
    # 2. Specular attenuation
    # -----------------------------------------

    spec_strength = np.mean(specular, axis=1)

    metal_loss = 1.0 - spec_strength

    # -----------------------------------------
    # 3. Roughness increase
    # -----------------------------------------

    rough_term = roughness

    # -----------------------------------------
    # 4. Final weather score
    # -----------------------------------------

    weather_score = (
        darkness * 0.35 +
        metal_loss * 0.25 +
        rough_term * 0.40
    )

    # normalize
    weather_score = (
        weather_score - weather_score.min()
    ) / (
        weather_score.max() -
        weather_score.min() +
        1e-8
    )

    return weather_score.astype(np.float32)

def reconstruct_semantic(
    samples_7d,
    embedded,
    query_points,
    sample_weather_score,
    trajectory_t,
    K=16,
    semantic_sigma=0.45,
    future_bias=0.9
):
    """
    samples_7d           : (N,7)
    embedded             : (N,3)
    query_points         : (M,3)

    sample_weather_score : (N,)
    trajectory_t         : (M,)

    return:
        reconstructed : (M,7)
    """

    N = samples_7d.shape[0]
    M = query_points.shape[0]

    out = np.zeros((M, 7), dtype=np.float32)

    # ----------------------------------------------------
    # 1. KNN search
    # ----------------------------------------------------

    nn = NearestNeighbors(
        n_neighbors=K * 4
    )

    nn.fit(embedded)

    dists, indices = nn.kneighbors(query_points)

    # ----------------------------------------------------
    # 2. Reconstruction
    # ----------------------------------------------------

    for qi in range(M):

        candidate_idx = indices[qi]
        candidate_dist = dists[qi]

        target_t = trajectory_t[qi]

        # ---------------------------------------------
        # semantic filtering
        # ---------------------------------------------

        candidate_weather = sample_weather_score[candidate_idx]

        semantic_delta = np.abs(
            candidate_weather - target_t
        )

        semantic_weight = np.exp(
            -(semantic_delta ** 2) /
            (2 * semantic_sigma * semantic_sigma)
        )

        # ---------------------------------------------
        # future bias
        # ---------------------------------------------

        future_mask = (
            candidate_weather >= target_t
        ).astype(np.float32)

        future_weight = (
            future_mask * future_bias +
            (1.0 - future_mask) * (1.0 - future_bias)
        )

        # ---------------------------------------------
        # spatial weight
        # ---------------------------------------------

        spatial_weight = np.exp(
            -candidate_dist * 4.0
        )

        # ---------------------------------------------
        # final weight
        # ---------------------------------------------

        weight = (
            semantic_weight *
            future_weight *
            spatial_weight
        )

        # stabilize
        weight += 1e-8

        weight /= np.sum(weight)

        # ---------------------------------------------
        # reconstruction
        # ---------------------------------------------

        reconstructed = np.sum(
            samples_7d[candidate_idx] *
            weight[:, None],
            axis=0
        )

        out[qi] = reconstructed

    # ----------------------------------------------------
    # 3. Clamp
    # ----------------------------------------------------

    out[:, :6] = np.clip(out[:, :6], 0, 1)
    out[:, 6] = np.clip(out[:, 6], 0, 1)

    return out


def build_weathering_sequence_semantic(
    samples_7d,
    embedded,
    trajectory,
    H,
    W,
    max_step=20,
    K=8
):

    diffuse = samples_7d[:, 0:3]
    specular = samples_7d[:, 3:6]
    roughness = samples_7d[:, 6]

    # ----------------------------------------------------
    # 1. Weather Score
    # ----------------------------------------------------

    weather_score = compute_weather_score_new(
        diffuse,
        specular,
        roughness
    )

    steps = []

    # ----------------------------------------------------
    # 2. trajectory parameterization
    # ----------------------------------------------------

    traj_t = np.linspace(
        0.0,
        1.0,
        len(trajectory)
    )

    # ----------------------------------------------------
    # 3. sequence generation
    # ----------------------------------------------------

    for step in range(max_step):

        alpha = (
            step / (max_step - 1)
        )

        # trajectory index
        tidx = int(
            alpha * (len(trajectory) - 1)
        )

        query_point = trajectory[tidx]

        # 모든 픽셀을 동일 semantic state로 이동
        trajectory_start = trajectory[0]
        trajectory_end   = trajectory[-1]

        flow = (
            trajectory_end -
            trajectory_start
        )

        query_points = (
            embedded +
            flow[None,:] * alpha
        )

        trajectory_t = np.full(
            samples_7d.shape[0],
            traj_t[tidx],
            dtype=np.float32
        )

        reconstructed = reconstruct_semantic(
            samples_7d=samples_7d,
            embedded=embedded,
            query_points=query_points,
            sample_weather_score=weather_score,
            trajectory_t=trajectory_t,
            K=K
        )

        steps.append(
            reconstructed.reshape(H, W, 7)
        )

    return steps

### Visualize Weathering sequence 

In [ ]:
steps = build_weathering_sequence_semantic(FX, Z, trajectory, H=128, W=128, max_step=40)
visualize_weathering_sequence(steps)

In [ ]:
def apply_weathering_physics(
    reconstructed,
    corrosion_mask,
    weather_t,
    roughness_strength=0.8,
    specular_decay=0.55,
    darkening_strength=0.25,
    desaturation_strength=0.18
):

    out = reconstructed.copy()

    # =====================================================
    # Roughness increase
    # =====================================================

    rough_delta = (
        corrosion_mask *
        weather_t *
        roughness_strength
    )

    out[:, 6] += rough_delta

    # =====================================================
    # Specular decay
    # =====================================================

    spec_scale = (
        1.0 -
        weather_t * specular_decay
    )

    out[:, 3:6] *= spec_scale

    # =====================================================
    # Diffuse darkening
    # =====================================================

    darkening = (
        1.0 -
        weather_t * darkening_strength
    )

    out[:, 0:3] *= darkening

    # =====================================================
    # Desaturation
    # =====================================================

    gray = np.mean(
        out[:, 0:3],
        axis=1,
        keepdims=True
    )

    blend = (
        desaturation_strength *
        weather_t
    )

    out[:, 0:3] = (
        out[:, 0:3] * (1.0 - blend) +
        gray * blend
    )

    # =====================================================
    # Clamp
    # =====================================================

    out[:, :6] = np.clip(
        out[:, :6],
        0,
        1
    )

    out[:, 6] = np.clip(
        out[:, 6],
        0,
        1
    )

    return out.astype(np.float32)


def generate_corrosion_mask(
    diffuse,
    roughness
):

    luminance = (
        diffuse[:,0] * 0.299 +
        diffuse[:,1] * 0.587 +
        diffuse[:,2] * 0.114
    )

    darkness = 1.0 - luminance

    mask = (
        darkness * 0.6 +
        roughness * 0.4
    )

    mask = (
        mask - mask.min()
    ) / (
        mask.max() -
        mask.min() +
        1e-8
    )

    return mask.astype(np.float32)

def build_weathering_sequence_physics(
    samples_7d,
    embedded,
    trajectory,
    H,
    W,
    max_step=20,
    K=16
):

    N = samples_7d.shape[0]

    diffuse = samples_7d[:, 0:3]
    specular = samples_7d[:, 3:6]
    roughness = samples_7d[:, 6]

    # =====================================================
    # Weather score
    # =====================================================

    weather_score = compute_weather_score_new(
        diffuse,
        specular,
        roughness
    )

    # =====================================================
    # Corrosion mask
    # =====================================================

    corrosion_mask = generate_corrosion_mask(
        diffuse,
        roughness
    )

    # =====================================================
    # Global trajectory flow
    # =====================================================

    flow = (
        trajectory[-1] -
        trajectory[0]
    )

    steps = []

    # =====================================================
    # Sequence generation
    # =====================================================

    for step in range(max_step):

        alpha = (
            step /
            (max_step - 1)
        )

        # -------------------------------------------------
        # Local manifold transport
        # -------------------------------------------------

        query_points = (
            embedded +
            flow[None, :] * alpha
        )

        trajectory_t = np.full(
            N,
            alpha,
            dtype=np.float32
        )

        # -------------------------------------------------
        # Semantic reconstruction
        # -------------------------------------------------

        reconstructed = reconstruct_semantic(
            samples_7d=samples_7d,
            embedded=embedded,
            query_points=query_points,
            sample_weather_score=weather_score,
            trajectory_t=trajectory_t,
            K=K
        )

        # -------------------------------------------------
        # Physics-guided material evolution
        # -------------------------------------------------

        reconstructed = apply_weathering_physics(
            reconstructed=reconstructed,
            corrosion_mask=corrosion_mask,
            weather_t=alpha
        )

        # -------------------------------------------------
        # Reshape
        # -------------------------------------------------

        steps.append(
            reconstructed.reshape(
                H,
                W,
                7
            )
        )

    return steps

In [ ]:
steps = build_weathering_sequence_physics(
    samples_7d=FX,
    embedded=Z,
    trajectory=trajectory,
    H=128,
    W=128,
    max_step=30,
    K=8
)

#for i, step in enumerate(steps):
#    visualize_step(step, i)

visualize_weathering_sequence(steps)